## Wczytanie danych i podział na train valid test

In [3]:
import os
import random
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
# --- 1. Globalne ziarno losowości ---
SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# --- 2. Wczytanie i złączenie danych ---
trans = pd.read_csv("dane/train_transaction.csv")
ident = pd.read_csv("dane/train_identity.csv")

df = trans.merge(ident, on="TransactionID", how="left")

# --- 3. Podział Out-Of-Time (chronologiczny) ---
df = df.sort_values("TransactionDT").reset_index(drop=True)

n = len(df)
n_train = int(n * 0.60)
n_valid = int(n * 0.80)   # 60% + 20%

train = df.iloc[:n_train]
valid = df.iloc[n_train:n_valid]
test  = df.iloc[n_valid:]

# --- 4. Weryfikacja podziału ---
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5s} | shape: {part.shape} | fraud rate: {part['isFraud'].mean():.4f}")


train | shape: (354324, 434) | fraud rate: 0.0338
valid | shape: (118108, 434) | fraud rate: 0.0390
test  | shape: (118108, 434) | fraud rate: 0.0344


## Data preprocessing

In [4]:
# Wymaga sklearn >= 1.3 (TargetEncoder) i scipy
import joblib
from scipy.stats import skew
from sklearn.feature_selection import VarianceThreshold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder,
                                    TargetEncoder, FunctionTransformer)
from sklearn.impute import SimpleImputer

# ── Stałe i grupy kolumn ─────────────────────────────────────────────────────
TARGET         = 'isFraud'
MISSING_THRESH = 0.80   # dla nie-V: odrzuć kolumny z NaN > 80%
V_CORR_THRESH  = 0.05   # dla V_xxx: zachowaj jeśli |corr(isFraud)| ≥ 0.05
VAR_THRESH     = 0.01   # odrzuć cechy numeryczne z wariancją < 0.01
SKEW_THRESH    = 3      # signed_log1p tylko dla |skewness| > 3
CARD_THRESH    = 15     # granica cat_low / cat_high

# Wykrywamy grupy kolumn ze struktury surowych danych
M_COLS  = [c for c in train.columns if c.startswith('M') and c[1:].isdigit()]
D_COLS  = [c for c in train.columns if c.startswith('D') and c[1:].isdigit()]
ID_COLS = [c for c in train.columns
           if c.startswith('id_') or c in ('DeviceType', 'DeviceInfo')]

# ── 1. Feature Engineering ───────────────────────────────────────────────────
def engineer_features(df):
    df = df.copy()

    # Czas z TransactionDT
    df['hour']      = (df['TransactionDT'] // 3600) % 24
    df['dayofweek'] = (df['TransactionDT'] // (3600 * 24)) % 7

    # Flaga: email nadawcy == email odbiorcy (fraud rate 3x wyższy gdy True wg EDA)
    df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(np.int64)

    # M1–M9: T→1, F→0, NaN→-1
    # NaN to odrębny stan semantyczny (niezweryfikowane), nie brakująca wartość
    for c in M_COLS:
        if c in df.columns:
            df[c] = df[c].map({'T': 1, 'F': 0}).fillna(-1).astype(np.float64)

    # Flagi missingness dla bloków D i identity:
    # sam brak wartości koreluje z fraudem (np. D7: 14.9% vs 2.7% wg EDA)
    for c in D_COLS + ID_COLS:
        if c in df.columns:
            df[f'{c}_isnan'] = df[c].isna().astype(np.int64)

    return df.drop(columns=['TransactionID', 'TransactionDT', TARGET], errors='ignore')

X_train = engineer_features(train);  y_train = train[TARGET].reset_index(drop=True)
X_valid = engineer_features(valid);  y_valid = valid[TARGET].reset_index(drop=True)
X_test  = engineer_features(test);   y_test  = test[TARGET].reset_index(drop=True)

# ── 2. Filtracja kolumn ──────────────────────────────────────────────────────
isnan_cols   = [c for c in X_train.columns if c.endswith('_isnan')]
v_cols       = [c for c in X_train.columns if c.startswith('V') and c[1:].isdigit()]
missing_rate = X_train.isna().mean()

# V_xxx: zachowaj jeśli NaN < 80% LUB |korelacja z isFraud| ≥ 0.05
# Ślepa filtracja 80% odrzucałaby 23 kolumny V o korelacji 0.05–0.28 (wg EDA)
v_corr = X_train[v_cols].corrwith(y_train).abs()
v_keep = [c for c in v_cols
          if missing_rate[c] < MISSING_THRESH or v_corr.get(c, 0) >= V_CORR_THRESH]

# Pozostałe (bez V_xxx i flag _isnan): standardowy próg 80%
# _isnan flagi zawsze zachowujemy – mają 0 NaN i niosą sygnał
other_cols = [c for c in X_train.columns if c not in v_cols + isnan_cols]
other_keep = [c for c in other_cols if missing_rate.get(c, 0) < MISSING_THRESH]

keep_cols = v_keep + other_keep + isnan_cols
X_train, X_valid, X_test = X_train[keep_cols], X_valid[keep_cols], X_test[keep_cols]
print(f"Kolumny po filtracji: {len(keep_cols)} "
      f"(V_xxx: {len(v_keep)}, inne: {len(other_keep)}, _isnan: {len(isnan_cols)})")

# ── 3. Grupy zmiennych, wariancja, skośność ──────────────────────────────────
# "Stable" = kolumny celowo wykluczone z VarianceThreshold i signed_log1p:
#   _isnan: binarne 0/1, semantycznie istotne nawet przy niskiej wariancji
#   M_COLS: pre-zakodowane {-1, 0, 1} – nie mają sensu logować
stable = set(isnan_cols) | {c for c in M_COLS if c in X_train.columns}

num_all = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_all = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

# Filtr wariancji (tylko nie-stable)
num_for_filter = [c for c in num_all if c not in stable]
selector = VarianceThreshold(threshold=VAR_THRESH)
selector.fit(X_train[num_for_filter])
low_var  = [c for c, ok in zip(num_for_filter, selector.get_support()) if not ok]

X_train = X_train.drop(columns=low_var)
X_valid = X_valid.drop(columns=low_var)
X_test  = X_test.drop(columns=low_var)
num_all = [c for c in num_all if c not in low_var]
print(f"Usunięto {len(low_var)} cech o niskiej wariancji → {len(num_all)} cech num")

# Skośność (tylko nie-stable) → signed_log1p kompresuje ogon bez wykrzywiania pozostałych
skewness   = X_train[[c for c in num_all if c not in stable]].apply(lambda c: skew(c.dropna()))
num_skew   = skewness[skewness.abs() > SKEW_THRESH].index.tolist()
num_normal = [c for c in num_all if c not in num_skew]  # zawiera stable

cat_low  = [c for c in cat_all if X_train[c].nunique() < CARD_THRESH]
cat_high = [c for c in cat_all if X_train[c].nunique() >= CARD_THRESH]

print(f"Numeryczne: {len(num_all)} (skośne: {len(num_skew)}, normalne+stable: {len(num_normal)})")
print(f"Kat. niska kard.: {len(cat_low)}  → {cat_low}")
print(f"Kat. wysoka kard.: {len(cat_high)} → {cat_high}")

# ── 4. ColumnTransformer ─────────────────────────────────────────────────────
def signed_log1p(x):
    # Kompresuje skośne rozkłady; działa dla x ujemnych (w odróżnieniu od log1p)
    return np.sign(x) * np.log1p(np.abs(x))

preprocessor = ColumnTransformer(transformers=[
    ('num_skew', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('log', FunctionTransformer(signed_log1p, validate=False)),
        ('scl', MinMaxScaler()),
    ]), num_skew),
    ('num_normal', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', MinMaxScaler()),
    ]), num_normal),
    ('cat_low', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='if_binary')),
    ]), cat_low),
    ('cat_high', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('enc', TargetEncoder(target_type='binary', smooth='auto')),  # kategoria → P(fraud)
        ('scl', MinMaxScaler()),
    ]), cat_high),
], remainder='drop')

# ── 5. Fit & Transform (fit tylko na train) ──────────────────────────────────
preprocessor.fit(X_train, y_train)

X_train_proc = preprocessor.transform(X_train).astype(np.float32)
X_valid_proc  = preprocessor.transform(X_valid).astype(np.float32)
X_test_proc   = preprocessor.transform(X_test).astype(np.float32)

assert not np.isnan(X_train_proc).any(), "NaN w X_train_proc!"
assert not np.isnan(X_valid_proc).any(),  "NaN w X_valid_proc!"
assert not np.isnan(X_test_proc).any(),   "NaN w X_test_proc!"

# ── 6. Filtrowanie pod Autoenkoder ───────────────────────────────────────────
# Sieć uczy się rekonstruować TYLKO normalne transakcje (y == 0).
# Fraudy będą miały wyższy błąd rekonstrukcji → próg detekcji anomalii.
X_train_ae = X_train_proc[y_train.values == 0]
X_valid_ae  = X_valid_proc   # pełny zbiór – ground truth: y_valid / y_test
X_test_ae   = X_test_proc

print(f"\nGotowe macierze:")
print(f"X_train_ae : {X_train_ae.shape}  ← tylko normalne transakcje")
print(f"X_valid_ae : {X_valid_ae.shape}")
print(f"X_test_ae  : {X_test_ae.shape}")
print(f"dtype      : {X_train_ae.dtype}")

# ── 7. Zapis ─────────────────────────────────────────────────────────────────
joblib.dump(preprocessor,             'preprocessor_ae.pkl')
joblib.dump(X_train.columns.tolist(), 'input_cols_ae.pkl')  # finalna lista kolumn wejściowych
print("\nZapisano: preprocessor_ae.pkl, input_cols_ae.pkl")


c:\Users\matma\Documents\SGH\SEM II\Data mining\Projekt_DM\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\matma\Documents\SGH\SEM II\Data mining\Projekt_DM\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Kolumny po filtracji: 444 (V_xxx: 321, inne: 68, _isnan: 55)
Usunięto 25 cech o niskiej wariancji → 402 cech num
Numeryczne: 402 (skośne: 247, normalne+stable: 155)
Kat. niska kard.: 13  → ['ProductCD', 'card4', 'card6', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']
Kat. wysoka kard.: 4 → ['P_emaildomain', 'R_emaildomain', 'id_31', 'DeviceInfo']

Gotowe macierze:
X_train_ae : (342336, 451)  ← tylko normalne transakcje
X_valid_ae : (118108, 451)
X_test_ae  : (118108, 451)
dtype      : float32

Zapisano: preprocessor_ae.pkl, input_cols_ae.pkl


## Trening modelu AutoEncoder

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# ── 1. DataLoadery ────────────────────────────────────────────────────────────
BATCH_SIZE = 512

X_train_t = torch.tensor(X_train_ae, dtype=torch.float32)
X_valid_t  = torch.tensor(X_valid_ae,  dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid_t),  batch_size=BATCH_SIZE, shuffle=False)

# ── 2. Architektura ───────────────────────────────────────────────────────────
INPUT_DIM  = 451
BOTTLENECK = 64   # v1: 32 → v2: 64 (więcej pojemności kodowania)

class FraudAutoEncoderV2(nn.Module):
    def __init__(self):
        super().__init__()
        act  = nn.LeakyReLU(0.1)
        drop = nn.Dropout(0.05)  # v1: 0.20 → v2: 0.05 (mniej regularyzacji)

        self.encoder = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), act, drop,
            nn.Linear(256, 128),       act,
            nn.Linear(128, BOTTLENECK), act,
        )
        self.decoder = nn.Sequential(
            nn.Linear(BOTTLENECK, 128), act, drop,
            nn.Linear(128, 256),        act, drop,
            nn.Linear(256, INPUT_DIM),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# ── 3. Konfiguracja ───────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Urządzenie: {device}")

model     = FraudAutoEncoderV2().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ── 4. Trening z Early Stopping ───────────────────────────────────────────────
MAX_EPOCHS = 150   # więcej epok – model może zbiegać wolniej przy niskim dropout
PATIENCE   = 15

best_val_loss     = float('inf')
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for (batch,) in train_loader:
        batch = batch.to(device)
        recon = model(batch)
        loss  = criterion(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch)
    train_loss /= len(X_train_t)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (batch,) in valid_loader:
            batch = batch.to(device)
            val_loss += criterion(model(batch), batch).item() * len(batch)
    val_loss /= len(X_valid_t)

    print(f"Epoch {epoch:3d}/{MAX_EPOCHS} | train={train_loss:.6f} | val={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_ae_model_v2.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping po {epoch} epokach (patience={PATIENCE}).")
            break

print(f"\nNajlepszy val_loss: {best_val_loss:.6f}")

# ── 5. Błędy rekonstrukcji na zbiorze walidacyjnym ────────────────────────────
model.load_state_dict(torch.load('best_ae_model_v2.pth', weights_only=True))
model.eval()

recon_errors = []
with torch.no_grad():
    for (batch,) in valid_loader:
        batch = batch.to(device)
        errors = ((model(batch) - batch) ** 2).mean(dim=1)
        recon_errors.append(errors.cpu().numpy())

valid_reconstruction_errors = np.concatenate(recon_errors)
print(f"valid_reconstruction_errors: shape={valid_reconstruction_errors.shape}, "
      f"min={valid_reconstruction_errors.min():.6f}, max={valid_reconstruction_errors.max():.6f}")


Urządzenie: cuda
Epoch   1/150 | train=0.017595 | val=0.008028
Epoch   2/150 | train=0.006120 | val=0.004338
Epoch   3/150 | train=0.003881 | val=0.003326
Epoch   4/150 | train=0.003039 | val=0.002531
Epoch   5/150 | train=0.002398 | val=0.002035
Epoch   6/150 | train=0.002008 | val=0.001748
Epoch   7/150 | train=0.001752 | val=0.001549
Epoch   8/150 | train=0.001593 | val=0.001428
Epoch   9/150 | train=0.001434 | val=0.001273
Epoch  10/150 | train=0.001297 | val=0.001167
Epoch  11/150 | train=0.001204 | val=0.001087
Epoch  12/150 | train=0.001131 | val=0.001024
Epoch  13/150 | train=0.001069 | val=0.000990
Epoch  14/150 | train=0.001017 | val=0.000939
Epoch  15/150 | train=0.000971 | val=0.000908
Epoch  16/150 | train=0.000928 | val=0.000860
Epoch  17/150 | train=0.000892 | val=0.000816
Epoch  18/150 | train=0.000856 | val=0.000790
Epoch  19/150 | train=0.000822 | val=0.000760
Epoch  20/150 | train=0.000796 | val=0.000714
Epoch  21/150 | train=0.000770 | val=0.000696
Epoch  22/150 | t

In [8]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
import numpy as np

fraud_errors  = valid_reconstruction_errors[y_valid == 1]
normal_errors = valid_reconstruction_errors[y_valid == 0]

print(f"Normalne — mediana: {np.median(normal_errors):.6f} | 95p: {np.percentile(normal_errors, 95):.6f} | 99p: {np.percentile(normal_errors, 99):.6f}")
print(f"Fraudy   — mediana: {np.median(fraud_errors):.6f} | 95p: {np.percentile(fraud_errors, 95):.6f} | 99p: {np.percentile(fraud_errors, 99):.6f}")

# Szukaj progu maksymalizującego F1
thresholds = np.linspace(valid_reconstruction_errors.min(), valid_reconstruction_errors.max(), 1000)
f1_scores  = [f1_score(y_valid, valid_reconstruction_errors >= t) for t in thresholds]

best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\nNajlepszy próg:  {best_threshold:.6f}")
print(f"F1 (walidacja):  {f1_scores[best_idx]:.4f}")
print(f"Precision:       {precision_score(y_valid, valid_reconstruction_errors >= best_threshold):.4f}")
print(f"Recall:          {recall_score(y_valid, valid_reconstruction_errors >= best_threshold):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_valid, valid_reconstruction_errors):.4f}")
print(f"AUPRC:    {average_precision_score(y_valid, valid_reconstruction_errors):.4f}")


Normalne — mediana: 0.000338 | 95p: 0.002481 | 99p: 0.004644
Fraudy   — mediana: 0.001486 | 95p: 0.009195 | 99p: 0.014891

Najlepszy próg:  0.003156
F1 (walidacja):  0.3073
Precision:       0.3204
Recall:          0.2952
ROC AUC:  0.7735
AUPRC:    0.2360


In [19]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve
)
import csv, os

# ── Architektury ──────────────────────────────────────────────────────────────
INPUT_DIM = 451

class FraudAutoEncoder(nn.Module):
    """v1: bottleneck=32, dropout=0.20"""
    def __init__(self):
        super().__init__()
        act  = nn.LeakyReLU(0.1)
        drop = nn.Dropout(0.20)
        self.encoder = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), act, drop,
            nn.Linear(256, 128),       act,
            nn.Linear(128, 32),        act,
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),        act, drop,
            nn.Linear(128, 256),       act, drop,
            nn.Linear(256, INPUT_DIM),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

class FraudAutoEncoderV2(nn.Module):
    """v2: bottleneck=64, dropout=0.05"""
    def __init__(self):
        super().__init__()
        act  = nn.LeakyReLU(0.1)
        drop = nn.Dropout(0.05)
        self.encoder = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), act, drop,
            nn.Linear(256, 128),       act,
            nn.Linear(128, 64),        act,
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),        act, drop,
            nn.Linear(128, 256),       act, drop,
            nn.Linear(256, INPUT_DIM),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# ── Funkcja ewaluacyjna ───────────────────────────────────────────────────────
def evaluate_autoencoder(
    model_class,
    weights_path,
    X_valid, y_valid,
    X_test,  y_test,
    run_meta: dict,            # run_id, bottleneck, dropout, epochs_trained, val_loss, notes
    csv_path='experiment_results.csv',
    batch_size=512,
    scoring='mse',             # 'mse' lub 'mae'
    device=None,
):
    """
    Ładuje model, liczy błędy rekonstrukcji, wyznacza próg na valid (max F1),
    raportuje metryki na valid i test, zapisuje wiersz do CSV.

    Zwraca dict z metrykami valid + test.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Załaduj model
    model = model_class().to(device)
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    model.eval()

    def _recon_errors(X):
        loader = DataLoader(
            TensorDataset(torch.tensor(X, dtype=torch.float32)),
            batch_size=batch_size, shuffle=False
        )
        errs = []
        with torch.no_grad():
            for (batch,) in loader:
                batch = batch.to(device)
                diff  = model(batch) - batch
                e = (diff ** 2).mean(dim=1) if scoring == 'mse' else diff.abs().mean(dim=1)
                errs.append(e.cpu().numpy())
        return np.concatenate(errs)

    scores_valid = _recon_errors(X_valid)
    scores_test  = _recon_errors(X_test)

    # Optymalny próg na valid
    prec, rec, thr = precision_recall_curve(y_valid, scores_valid)
    f1_grid  = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_idx = np.argmax(f1_grid)
    best_thr = thr[best_idx]

    # Metryki valid
    yp_v  = (scores_valid >= best_thr).astype(int)
    v = {
        'F1':       f1_score(y_valid, yp_v),
        'Precision':precision_score(y_valid, yp_v),
        'Recall':   recall_score(y_valid, yp_v),
        'ROC_AUC':  roc_auc_score(y_valid, scores_valid),
        'AUPRC':    average_precision_score(y_valid, scores_valid),
    }
    v['Gini'] = 2 * v['ROC_AUC'] - 1

    # Metryki test
    yp_t  = (scores_test >= best_thr).astype(int)
    t = {
        'F1':       f1_score(y_test, yp_t),
        'Precision':precision_score(y_test, yp_t),
        'Recall':   recall_score(y_test, yp_t),
        'ROC_AUC':  roc_auc_score(y_test, scores_test),
        'AUPRC':    average_precision_score(y_test, scores_test),
    }
    t['Gini'] = 2 * t['ROC_AUC'] - 1

    # Wydruk
    label = weights_path
    print(f"\n{'='*55}")
    print(f"  Model: {label}   (próg={best_thr:.6f})")
    print(f"{'='*55}")
    print(f"{'Metryka':<12} {'VALID':>8} {'TEST':>8}")
    print(f"{'-'*30}")
    for key in ('F1','Precision','Recall','ROC_AUC','Gini','AUPRC'):
        print(f"{key:<12} {v[key]:>8.4f} {t[key]:>8.4f}")

    # Zapis do CSV
    fieldnames = ['run_id','model','bottleneck','dropout','loss_fn','epochs_trained',
                  'val_loss','scoring_method','threshold_strategy',
                  'F1','Precision','Recall','ROC_AUC','Gini','AUPRC','notes']
    row = {
        'run_id':             run_meta.get('run_id'),
        'model':              model_class.__name__,
        'bottleneck':         run_meta.get('bottleneck'),
        'dropout':            run_meta.get('dropout'),
        'loss_fn':            run_meta.get('loss_fn', 'MSELoss'),
        'epochs_trained':     run_meta.get('epochs_trained'),
        'val_loss':           run_meta.get('val_loss'),
        'scoring_method':     f'mean_{scoring.upper()}',
        'threshold_strategy': 'max_F1_on_valid',
        'F1':        round(v['F1'], 4),
        'Precision': round(v['Precision'], 4),
        'Recall':    round(v['Recall'], 4),
        'ROC_AUC':   round(v['ROC_AUC'], 4),
        'Gini':      round(v['Gini'], 4),
        'AUPRC':     round(v['AUPRC'], 4),
        'notes':     run_meta.get('notes', '') +
                     f" | test_F1={t['F1']:.4f} test_AUC={t['ROC_AUC']:.4f} test_Gini={t['Gini']:.4f}",
    }
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"\n  → zapisano do {csv_path} (run_id={row['run_id']})")

    return {'valid': v, 'test': t, 'threshold': best_thr,
            'scores_valid': scores_valid, 'scores_test': scores_test}

In [20]:

# ── Wywołania ─────────────────────────────────────────────────────────────────
results_v1_mse = evaluate_autoencoder(
    model_class   = FraudAutoEncoder,
    weights_path  = 'best_ae_model.pth',
    X_valid=X_valid_ae, y_valid=y_valid,
    X_test=X_test_ae,   y_test=y_test,
    scoring='mse',
    run_meta = {
        'run_id': 1, 'bottleneck': 32, 'dropout': 0.20,
        'epochs_trained': 100, 'val_loss': 0.000723,
        'notes': 'baseline v1 MSE',
    },
)

results_v2_mse = evaluate_autoencoder(
    model_class   = FraudAutoEncoderV2,
    weights_path  = 'best_ae_model_v2.pth',
    X_valid=X_valid_ae, y_valid=y_valid,
    X_test=X_test_ae,   y_test=y_test,
    scoring='mse',
    run_meta = {
        'run_id': 2, 'bottleneck': 64, 'dropout': 0.05,
        'epochs_trained': 150, 'val_loss': 0.000260,
        'notes': 'bottleneck=64 dropout=0.05 MSE',
    },
)

results_v1_mae = evaluate_autoencoder(
    model_class   = FraudAutoEncoder,
    weights_path  = 'best_ae_model.pth',
    X_valid=X_valid_ae, y_valid=y_valid,
    X_test=X_test_ae,   y_test=y_test,
    scoring='mae',
    run_meta = {
        'run_id': 4, 'bottleneck': 32, 'dropout': 0.20,
        'epochs_trained': 100, 'val_loss': 0.000723,
        'notes': 'baseline v1 MAE',
    },
)

results_v2_mae = evaluate_autoencoder(
    model_class   = FraudAutoEncoderV2,
    weights_path  = 'best_ae_model_v2.pth',
    X_valid=X_valid_ae, y_valid=y_valid,
    X_test=X_test_ae,   y_test=y_test,
    scoring='mae',
    run_meta = {
        'run_id': 5, 'bottleneck': 64, 'dropout': 0.05,
        'epochs_trained': 150, 'val_loss': 0.000260,
        'notes': 'bottleneck=64 dropout=0.05 MAE',
    },
)


  Model: best_ae_model.pth   (próg=0.002979)
Metryka         VALID     TEST
------------------------------
F1             0.3077   0.2284
Precision      0.3006   0.1886
Recall         0.3151   0.2896
ROC_AUC        0.7735   0.7625
Gini           0.5471   0.5249
AUPRC          0.2360   0.1155

  → zapisano do experiment_results.csv (run_id=1)

  Model: best_ae_model_v2.pth   (próg=0.000835)
Metryka         VALID     TEST
------------------------------
F1             0.3007   0.2403
Precision      0.2514   0.1841
Recall         0.3741   0.3460
ROC_AUC        0.7729   0.7618
Gini           0.5458   0.5235
AUPRC          0.2060   0.1160

  → zapisano do experiment_results.csv (run_id=2)

  Model: best_ae_model.pth   (próg=0.019125)
Metryka         VALID     TEST
------------------------------
F1             0.3240   0.2514
Precision      0.3172   0.2069
Recall         0.3312   0.3201
ROC_AUC        0.7834   0.7725
Gini           0.5669   0.5449
AUPRC          0.2542   0.1275

  → zapisano

## Teraz wytrenujemy sieć neuronową jako benchmark

In [33]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve
)
import csv, os

# ── Architektura ──────────────────────────────────────────────────────────────
INPUT_DIM = 451

def _block(in_dim, out_dim, dropout):
    # Pre-norm: Linear → LayerNorm → ReLU → Dropout
    return nn.Sequential(
        nn.Linear(in_dim, out_dim),
        nn.LayerNorm(out_dim),
        nn.ReLU(),
        nn.Dropout(dropout),
    )

class FraudMLP(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            _block(input_dim, 256, dropout),
            _block(256, 128, dropout),
            _block(128, 64,  dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


# ── Trening ───────────────────────────────────────────────────────────────────
def train_mlp(
    X_train, y_train,
    X_valid, y_valid,
    weights_path='best_mlp_model.pth',
    max_epochs=100,
    patience=10,
    batch_size=512,
    lr=1e-3,
    dropout=0.15,
    device=None,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Urządzenie: {device}")

    # pos_weight kompensuje niezbalansowanie klas (~3.4% fraudów)
    # clamp do 10 – surowe ~28 destabilizuje trening na początku
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    pos_w = torch.tensor([n_neg / n_pos], dtype=torch.float32).clamp(max=10.0).to(device)
    print(f"Niezbalansowanie: {n_neg} normalnych / {n_pos} fraudów → pos_weight={pos_w.item():.1f} (capped)")

    X_tr_t = torch.tensor(X_train, dtype=torch.float32)
    y_tr_t = torch.tensor(y_train.values if hasattr(y_train, 'values') else y_train,
                           dtype=torch.float32)
    X_va_t = torch.tensor(X_valid, dtype=torch.float32)
    y_va_t = torch.tensor(y_valid.values if hasattr(y_valid, 'values') else y_valid,
                           dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                              batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(TensorDataset(X_va_t, y_va_t),
                              batch_size=batch_size, shuffle=False)

    model     = FraudMLP(dropout=dropout).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    best_auc          = 0.0
    best_val_loss     = float('inf')
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            loss = criterion(model(X_b), y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(X_b)
        train_loss /= len(X_tr_t)

        model.eval()
        val_loss  = 0.0
        val_probs = []
        val_true  = []
        with torch.no_grad():
            for X_b, y_b in valid_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits    = model(X_b)
                val_loss += criterion(logits, y_b).item() * len(X_b)
                val_probs.append(torch.sigmoid(logits).cpu().numpy())
                val_true.append(y_b.cpu().numpy())
        val_loss /= len(X_va_t)
        val_auc   = roc_auc_score(np.concatenate(val_true), np.concatenate(val_probs))

        scheduler.step(val_loss)
        print(f"Epoch {epoch:3d}/{max_epochs} | train={train_loss:.6f} | val={val_loss:.6f} | val_AUC={val_auc:.4f}")

        if val_auc > best_auc:
            best_auc          = val_auc
            best_val_loss     = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), weights_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping po {epoch} epokach (best AUC={best_auc:.4f}).")
                break

    print(f"\nNajlepszy val AUC: {best_auc:.4f} | val_loss: {best_val_loss:.6f}")
    return epoch, best_val_loss

In [34]:
# ── Ewaluacja ─────────────────────────────────────────────────────────────────
def evaluate_classifier(
    weights_path,
    X_valid, y_valid,
    X_test,  y_test,
    run_meta: dict,
    csv_path='experiment_results.csv',
    batch_size=512,
    dropout=0.15,
    device=None,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = FraudMLP(dropout=dropout).to(device)
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    model.eval()

    def _proba(X):
        loader = DataLoader(
            TensorDataset(torch.tensor(X, dtype=torch.float32)),
            batch_size=batch_size, shuffle=False
        )
        probs = []
        with torch.no_grad():
            for (batch,) in loader:
                logits = model(batch.to(device))
                probs.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(probs)

    scores_valid = _proba(X_valid)
    scores_test  = _proba(X_test)

    # Optymalny próg na valid (max F1)
    prec, rec, thr = precision_recall_curve(y_valid, scores_valid)
    f1_grid  = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_idx = np.argmax(f1_grid)
    best_thr = thr[best_idx]

    y_valid_arr = y_valid.values if hasattr(y_valid, 'values') else y_valid
    y_test_arr  = y_test.values  if hasattr(y_test,  'values') else y_test

    yp_v = (scores_valid >= best_thr).astype(int)
    v = {
        'F1':        f1_score(y_valid_arr, yp_v),
        'Precision': precision_score(y_valid_arr, yp_v),
        'Recall':    recall_score(y_valid_arr, yp_v),
        'ROC_AUC':   roc_auc_score(y_valid_arr, scores_valid),
        'AUPRC':     average_precision_score(y_valid_arr, scores_valid),
    }
    v['Gini'] = 2 * v['ROC_AUC'] - 1

    yp_t = (scores_test >= best_thr).astype(int)
    t = {
        'F1':        f1_score(y_test_arr, yp_t),
        'Precision': precision_score(y_test_arr, yp_t),
        'Recall':    recall_score(y_test_arr, yp_t),
        'ROC_AUC':   roc_auc_score(y_test_arr, scores_test),
        'AUPRC':     average_precision_score(y_test_arr, scores_test),
    }
    t['Gini'] = 2 * t['ROC_AUC'] - 1

    print(f"\n{'='*55}")
    print(f"  Model: {weights_path}   (próg={best_thr:.4f})")
    print(f"{'='*55}")
    print(f"{'Metryka':<12} {'VALID':>8} {'TEST':>8}")
    print(f"{'-'*30}")
    for key in ('F1', 'Precision', 'Recall', 'ROC_AUC', 'Gini', 'AUPRC'):
        print(f"{key:<12} {v[key]:>8.4f} {t[key]:>8.4f}")

    fieldnames = ['run_id','model','bottleneck','dropout','loss_fn','epochs_trained',
                  'val_loss','scoring_method','threshold_strategy',
                  'F1','Precision','Recall','ROC_AUC','Gini','AUPRC','notes']
    row = {
        'run_id':             run_meta.get('run_id'),
        'model':              'FraudMLP',
        'bottleneck':         run_meta.get('bottleneck', '-'),
        'dropout':            run_meta.get('dropout', dropout),
        'loss_fn':            run_meta.get('loss_fn', 'BCEWithLogitsLoss+pos_weight'),
        'epochs_trained':     run_meta.get('epochs_trained'),
        'val_loss':           round(run_meta.get('val_loss', 0), 6),
        'scoring_method':     'sigmoid_prob',
        'threshold_strategy': 'max_F1_on_valid',
        'F1':        round(v['F1'], 4),
        'Precision': round(v['Precision'], 4),
        'Recall':    round(v['Recall'], 4),
        'ROC_AUC':   round(v['ROC_AUC'], 4),
        'Gini':      round(v['Gini'], 4),
        'AUPRC':     round(v['AUPRC'], 4),
        'notes':     run_meta.get('notes', '') +
                     f" | test_F1={t['F1']:.4f} test_AUC={t['ROC_AUC']:.4f} test_Gini={t['Gini']:.4f}",
    }
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"\n  → zapisano do {csv_path} (run_id={row['run_id']})")

    return {'valid': v, 'test': t, 'threshold': best_thr,
            'scores_valid': scores_valid, 'scores_test': scores_test}

In [35]:
# ── Wywołanie ─────────────────────────────────────────────────────────────────
# X_train_proc – pełny zbiór treningowy (354k, normalne + fraudy)
# y_train      – etykiety (0/1)
# X_valid_ae, X_test_ae, y_valid, y_test – z preprocessingu

epochs_trained, best_val = train_mlp(
    X_train=X_train_proc, y_train=y_train,
    X_valid=X_valid_ae,   y_valid=y_valid,
    weights_path='best_mlp_v3_model.pth',
    max_epochs=100,
    patience=15,
    lr=5e-4,
    dropout=0.15,
)

Urządzenie: cuda
Niezbalansowanie: 342336 normalnych / 11988 fraudów → pos_weight=10.0 (capped)
Epoch   1/100 | train=0.517313 | val=0.503031 | val_AUC=0.8767
Epoch   2/100 | train=0.463348 | val=0.495348 | val_AUC=0.8807
Epoch   3/100 | train=0.443064 | val=0.544823 | val_AUC=0.8681
Epoch   4/100 | train=0.434807 | val=0.492155 | val_AUC=0.8833
Epoch   5/100 | train=0.426192 | val=0.508626 | val_AUC=0.8911
Epoch   6/100 | train=0.421322 | val=0.474086 | val_AUC=0.8898
Epoch   7/100 | train=0.416108 | val=0.495602 | val_AUC=0.8915
Epoch   8/100 | train=0.410430 | val=0.476006 | val_AUC=0.8869
Epoch   9/100 | train=0.414112 | val=0.482859 | val_AUC=0.8853
Epoch  10/100 | train=0.404901 | val=0.503978 | val_AUC=0.8914
Epoch  11/100 | train=0.401858 | val=0.468338 | val_AUC=0.8932
Epoch  12/100 | train=0.398334 | val=0.490795 | val_AUC=0.8861
Epoch  13/100 | train=0.397194 | val=0.473548 | val_AUC=0.8893
Epoch  14/100 | train=0.394759 | val=0.492484 | val_AUC=0.8807
Epoch  15/100 | train=

In [36]:
results_mlp_v3 = evaluate_classifier(
    weights_path='best_mlp_v3_model.pth',
    X_valid=X_valid_ae, y_valid=y_valid,
    X_test=X_test_ae,   y_test=y_test,
    run_meta={
        'run_id': 7,
        'dropout': 0.15,
        'epochs_trained': epochs_trained,
        'val_loss': best_val,
        'notes': 'supervised MLP v3 | 451→256→128→64→1 LN+dropout=0.15 pos_w_cap10 lr=5e-4 patience=15 ES=AUC',
    },
)


  Model: best_mlp_v3_model.pth   (próg=0.7910)
Metryka         VALID     TEST
------------------------------
F1             0.5221   0.4447
Precision      0.5847   0.4750
Recall         0.4717   0.4181
ROC_AUC        0.8932   0.8610
Gini           0.7865   0.7221
AUPRC          0.5349   0.4199

  → zapisano do experiment_results.csv (run_id=7)


## Podejście Hybrydowe: Denoising AE -> XGBoost

In [47]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve
)
import csv, os

# ── 1. Architektura Denoising AE ──────────────────────────────────────────────
INPUT_DIM  = 451
BOTTLENECK = 64

class DenoisingAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        act  = nn.LeakyReLU(0.1)
        drop = nn.Dropout(0.05)

        self.encoder = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), act, drop,
            nn.Linear(256, 128),       act,
            nn.Linear(128, BOTTLENECK), act,
        )
        self.decoder = nn.Sequential(
            nn.Linear(BOTTLENECK, 128), act, drop,
            nn.Linear(128, 256),        act, drop,
            nn.Linear(256, INPUT_DIM),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def encode(self, x):
        return self.encoder(x)


# ── 2. SwapNoise – zamiana wartości między wierszami batcha ───────────────────
def swap_noise(X_batch, noise_rate=0.15):
    """
    Dla każdej komórki (i, j): z prawdopodobieństwem noise_rate zastąp wartość
    wartością z losowego wiersza w batchu (tej samej kolumny j).
    Zmusza sieć do nauki korelacji między cechami – silniejsze niż masking.
    """
    X_noisy = X_batch.clone()
    n, d = X_batch.shape
    mask = torch.bernoulli(torch.ones(n, d, device=X_batch.device) * noise_rate).bool()
    rand_rows = torch.randint(0, n, (n, d), device=X_batch.device)
    col_idx   = torch.arange(d, device=X_batch.device).unsqueeze(0).expand(n, d)
    swapped   = X_batch[rand_rows, col_idx]
    X_noisy[mask] = swapped[mask]
    return X_noisy


# ── 3. Trening DAE ────────────────────────────────────────────────────────────
def train_dae(
    X_all,                        # WSZYSTKIE transakcje (train+valid+test) – bez filtra isFraud
    X_valid,                      # zbiór walidacyjny (do early stopping)
    weights_path='best_dae_model_v2.pth',
    noise_rate=0.15,
    max_epochs=150,
    patience=15,
    batch_size=512,
    device=None,
):
    """
    DAE jako Feature Generator:
    - trenujemy na pełnym zbiorze treningowym (wszystkie klasy, bez danych testowych)
    - SwapNoise zamiast Maskingu – wymusza naukę korelacji między cechami
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Urządzenie: {device}")
    print(f"Rozmiar X_all: {X_all.shape} | noise_rate={noise_rate}")

    X_all_t = torch.tensor(X_all,   dtype=torch.float32)
    X_va_t  = torch.tensor(X_valid, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_all_t), batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(TensorDataset(X_va_t),  batch_size=batch_size, shuffle=False)

    model     = DenoisingAutoEncoder().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss     = float('inf')
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for (batch,) in train_loader:
            batch       = batch.to(device)
            noisy_batch = swap_noise(batch, noise_rate)
            recon       = model(noisy_batch)
            loss        = criterion(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(batch)
        train_loss /= len(X_all_t)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (batch,) in valid_loader:
                batch     = batch.to(device)
                val_loss += criterion(model(swap_noise(batch, noise_rate)), batch).item() * len(batch)
        val_loss /= len(X_va_t)

        print(f"Epoch {epoch:3d}/{max_epochs} | train={train_loss:.6f} | val={val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), weights_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping po {epoch} epokach.")
                break

    print(f"\nNajlepszy val_loss: {best_val_loss:.6f}")
    return epoch, best_val_loss
# ── 4. Ekstrakcja cech: latent + residuały per cecha ─────────────────────────
def extract_features_v2(X, weights_path, batch_size=512, device=None):
    """
    Zwraca (latent, residuals_per_feature):
      latent:               (n, 64)  – bottleneck DAE
      residuals_per_feature:(n, 451) – abs(X - X_hat) per kolumna
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = DenoisingAutoEncoder().to(device)
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    model.eval()

    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size, shuffle=False
    )

    latents, residuals = [], []
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            z     = model.encode(batch)
            recon = model.decoder(z)
            latents.append(z.cpu().numpy())
            residuals.append((recon - batch).abs().cpu().numpy())   # (b, 451)

    return np.concatenate(latents), np.concatenate(residuals)


# ── 5. Budowanie rozszerzonego zbioru cech 966D ───────────────────────────────
def build_augmented_v2(X, weights_path, batch_size=512, device=None):
    """
    X_aug = [raw 451 | DAE latent 64 | per-feature residuals 451] = 966 cech.

    Residuały per cecha (nie skalar) dają XGBoostowi informację o tym,
    które konkretne cechy transakcji są 'podejrzane' wg DAE.
    """
    latent, residuals = extract_features_v2(X, weights_path, batch_size, device)
    return np.concatenate([X, latent, residuals], axis=1).astype(np.float32)


# ── 6. Ogólna funkcja ewaluacji ───────────────────────────────────────────────
def evaluate_scores(
    scores_valid, scores_test,
    y_valid, y_test,
    run_meta: dict,
    csv_path='experiment_results.csv',
):
    y_valid_arr = y_valid.values if hasattr(y_valid, 'values') else y_valid
    y_test_arr  = y_test.values  if hasattr(y_test,  'values') else y_test

    prec, rec, thr = precision_recall_curve(y_valid_arr, scores_valid)
    f1_grid  = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_thr = thr[np.argmax(f1_grid)]

    def _metrics(scores, y_true, thr):
        yp  = (scores >= thr).astype(int)
        auc = roc_auc_score(y_true, scores)
        return {
            'F1':        f1_score(y_true, yp),
            'Precision': precision_score(y_true, yp),
            'Recall':    recall_score(y_true, yp),
            'ROC_AUC':   auc,
            'Gini':      2 * auc - 1,
            'AUPRC':     average_precision_score(y_true, scores),
        }

    v = _metrics(scores_valid, y_valid_arr, best_thr)
    t = _metrics(scores_test,  y_test_arr,  best_thr)

    print(f"\n{'='*55}")
    print(f"  Model: {run_meta.get('model','?')}   (próg={best_thr:.4f})")
    print(f"{'='*55}")
    print(f"{'Metryka':<12} {'VALID':>8} {'TEST':>8}")
    print(f"{'-'*30}")
    for key in ('F1', 'Precision', 'Recall', 'ROC_AUC', 'Gini', 'AUPRC'):
        print(f"{key:<12} {v[key]:>8.4f} {t[key]:>8.4f}")

    fieldnames = ['run_id','model','bottleneck','dropout','loss_fn','epochs_trained',
                  'val_loss','scoring_method','threshold_strategy',
                  'F1','Precision','Recall','ROC_AUC','Gini','AUPRC','notes']
    row = {
        'run_id':             run_meta.get('run_id'),
        'model':              run_meta.get('model'),
        'bottleneck':         run_meta.get('bottleneck', BOTTLENECK),
        'dropout':            run_meta.get('dropout', '-'),
        'loss_fn':            run_meta.get('loss_fn', '-'),
        'epochs_trained':     run_meta.get('epochs_trained', '-'),
        'val_loss':           run_meta.get('val_loss', '-'),
        'scoring_method':     run_meta.get('scoring_method', 'xgb_proba'),
        'threshold_strategy': 'max_F1_on_valid',
        'F1':        round(v['F1'], 4),
        'Precision': round(v['Precision'], 4),
        'Recall':    round(v['Recall'], 4),
        'ROC_AUC':   round(v['ROC_AUC'], 4),
        'Gini':      round(v['Gini'], 4),
        'AUPRC':     round(v['AUPRC'], 4),
        'notes':     run_meta.get('notes', '') +
                     f" | test_F1={t['F1']:.4f} test_AUC={t['ROC_AUC']:.4f} test_Gini={t['Gini']:.4f}",
    }
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"\n  → zapisano do {csv_path} (run_id={row['run_id']})")

    return {'valid': v, 'test': t, 'threshold': best_thr}

In [48]:
# ── 7. Wywołanie ──────────────────────────────────────────────────────────────
# Zmienne z notebooka: X_train_ae, X_valid_ae, X_test_ae, X_train_proc,
#                      y_train, y_valid, y_test

# ── 7a. Trening DAE na pełnym zbiorze treningowym (Feature Generator) ────────
# Wszystkie transakcje treningowe (normalne + fraudy) bez danych testowych.
# Usunięcie filtra isFraud==0: DAE jako generator cech, nie detektor anomalii.
X_all_for_dae = X_train_proc
print(f"X_all_for_dae: {X_all_for_dae.shape}  (X_train_proc, wszystkie klasy)")

dae_epochs, dae_val_loss = train_dae(
    X_all        = X_all_for_dae,
    X_valid      = X_valid_ae,
    weights_path = 'best_dae_model_v2.pth',
    noise_rate   = 0.15,
    max_epochs   = 150,
    patience     = 15,
)

X_all_for_dae: (354324, 451)  (X_train_proc, wszystkie klasy)
Urządzenie: cuda
Rozmiar X_all: (354324, 451) | noise_rate=0.15
Epoch   1/150 | train=0.017808 | val=0.008062
Epoch   2/150 | train=0.006690 | val=0.005377
Epoch   3/150 | train=0.004897 | val=0.004354
Epoch   4/150 | train=0.004257 | val=0.003913
Epoch   5/150 | train=0.003818 | val=0.003505
Epoch   6/150 | train=0.003469 | val=0.003160
Epoch   7/150 | train=0.003208 | val=0.002825
Epoch   8/150 | train=0.003001 | val=0.002697
Epoch   9/150 | train=0.002847 | val=0.002583
Epoch  10/150 | train=0.002725 | val=0.002407
Epoch  11/150 | train=0.002624 | val=0.002377
Epoch  12/150 | train=0.002548 | val=0.002365
Epoch  13/150 | train=0.002475 | val=0.002294
Epoch  14/150 | train=0.002412 | val=0.002174
Epoch  15/150 | train=0.002363 | val=0.002124
Epoch  16/150 | train=0.002315 | val=0.002193
Epoch  17/150 | train=0.002275 | val=0.002132
Epoch  18/150 | train=0.002248 | val=0.002046
Epoch  19/150 | train=0.002210 | val=0.002036


In [49]:
# ── 7b. Budowanie cech 966D (451 raw + 64 latent + 451 residuals) ─────────────
print("\nEkstrakcja cech z DAE v2 (966D)...")
X_train_aug = build_augmented_v2(X_train_proc, 'best_dae_model_v2.pth')
X_valid_aug = build_augmented_v2(X_valid_ae,   'best_dae_model_v2.pth')
X_test_aug  = build_augmented_v2(X_test_ae,    'best_dae_model_v2.pth')
print(f"X_train_aug: {X_train_aug.shape}   (oczekiwane: {X_train_proc.shape[0]} × 966)")
print(f"X_valid_aug: {X_valid_aug.shape}")


Ekstrakcja cech z DAE v2 (966D)...
X_train_aug: (354324, 966)   (oczekiwane: 354324 × 966)
X_valid_aug: (118108, 966)


In [50]:
# ── 7c. Trening XGBoost v2 ───────────────────────────────────────────────────
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
spw   = n_neg / n_pos
print(f"\nXGBoost v2 | scale_pos_weight={spw:.1f} | features=966")

y_train_arr = y_train.values if hasattr(y_train, 'values') else y_train
y_valid_arr = y_valid.values if hasattr(y_valid, 'values') else y_valid

xgb_model_v2 = xgb.XGBClassifier(
    n_estimators          = 2000,
    max_depth             = 7,
    learning_rate         = 0.02,
    subsample             = 0.8,
    colsample_bytree      = 0.3,      # kluczowe przy ~1000 cechach
    min_child_weight      = 5,
    scale_pos_weight      = spw,
    eval_metric           = 'aucpr',  # lepiej koreluje z F1 niż logloss
    early_stopping_rounds = 100,
    device                = 'cuda',
    random_state          = 123,
    verbosity             = 1,
)

xgb_model_v2.fit(
    X_train_aug, y_train_arr,
    eval_set=[(X_valid_aug, y_valid_arr)],
    verbose=100,
)


XGBoost v2 | scale_pos_weight=28.6 | features=966
[0]	validation_0-aucpr:0.32576
[100]	validation_0-aucpr:0.47676
[200]	validation_0-aucpr:0.48594
[300]	validation_0-aucpr:0.49195
[400]	validation_0-aucpr:0.49593
[500]	validation_0-aucpr:0.49968
[600]	validation_0-aucpr:0.50237
[700]	validation_0-aucpr:0.50604
[800]	validation_0-aucpr:0.50973
[900]	validation_0-aucpr:0.51230
[1000]	validation_0-aucpr:0.51408
[1100]	validation_0-aucpr:0.51592
[1200]	validation_0-aucpr:0.51747
[1300]	validation_0-aucpr:0.51880
[1400]	validation_0-aucpr:0.52027
[1500]	validation_0-aucpr:0.52204
[1600]	validation_0-aucpr:0.52326
[1700]	validation_0-aucpr:0.52446
[1800]	validation_0-aucpr:0.52489
[1900]	validation_0-aucpr:0.52508
[1999]	validation_0-aucpr:0.52665


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.3
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",100
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [51]:
# ── 7d. Ewaluacja ─────────────────────────────────────────────────────────────
scores_valid = xgb_model_v2.predict_proba(X_valid_aug)[:, 1]
scores_test  = xgb_model_v2.predict_proba(X_test_aug)[:, 1]

results_dae_xgb_v2 = evaluate_scores(
    scores_valid = scores_valid,
    scores_test  = scores_test,
    y_valid      = y_valid,
    y_test       = y_test,
    run_meta = {
        'run_id':         10,
        'model':          'DAE_v2+XGBoost_v2',
        'bottleneck':     BOTTLENECK,
        'loss_fn':        'MSELoss(DAE,SwapNoise)+XGB(aucpr)',
        'epochs_trained': dae_epochs,
        'val_loss':       round(dae_val_loss, 6),
        'scoring_method': 'xgb_proba',
        'notes': (
            f'DAE SwapNoise=0.15 | train_on=all({X_all_for_dae.shape[0]}) | '
            f'XGB n_est={xgb_model_v2.best_iteration} | '
            f'aug=451+64+451=966 | colsample=0.3 | lr=0.02'
        ),
    },
)


  Model: DAE_v2+XGBoost_v2   (próg=0.5519)
Metryka         VALID     TEST
------------------------------
F1             0.5179   0.4421
Precision      0.5551   0.4722
Recall         0.4854   0.4156
ROC_AUC        0.8946   0.8626
Gini           0.7892   0.7252
AUPRC          0.5267   0.4508

  → zapisano do experiment_results.csv (run_id=10)


In [46]:
# ── 8. Czysty XGBoost (bez DAE) – punkt odniesienia ─────────────────────────
print("\n=== Czysty XGBoost na oryginalnych cechach (451) ===")

xgb_pure = xgb.XGBClassifier(
    n_estimators          = 1000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    scale_pos_weight      = n_neg / n_pos,
    eval_metric           = 'auc',
    early_stopping_rounds = 50,
    device                = 'cuda',
    random_state          = 123,
    verbosity             = 1,
)

xgb_pure.fit(
    X_train_proc, y_train_arr,
    eval_set=[(X_valid_ae, y_valid_arr)],
    verbose=50,
)

scores_valid_pure = xgb_pure.predict_proba(X_valid_ae)[:, 1]
scores_test_pure  = xgb_pure.predict_proba(X_test_ae)[:, 1]

results_xgb_pure = evaluate_scores(
    scores_valid = scores_valid_pure,
    scores_test  = scores_test_pure,
    y_valid      = y_valid,
    y_test       = y_test,
    run_meta = {
        'run_id':         9,
        'model':          'XGBoost_pure',
        'bottleneck':     '-',
        'loss_fn':        'XGB',
        'scoring_method': 'xgb_proba',
        'notes':          f'czysty XGB na 451 cechach | n_est={xgb_pure.best_iteration}',
    },
)



=== Czysty XGBoost na oryginalnych cechach (451) ===
[0]	validation_0-auc:0.77938
[50]	validation_0-auc:0.85156
[100]	validation_0-auc:0.85768
[132]	validation_0-auc:0.85832

  Model: XGBoost_pure   (próg=0.7799)
Metryka         VALID     TEST
------------------------------
F1             0.4641   0.4099
Precision      0.5508   0.4602
Recall         0.4010   0.3696
ROC_AUC        0.8594   0.8437
Gini           0.7187   0.6874
AUPRC          0.4568   0.4108

  → zapisano do experiment_results.csv (run_id=9)


In [45]:
display(pd.read_csv("experiment_results.csv"))

,run_id,model,bottleneck,dropout,loss_fn,epochs_trained,val_loss,scoring_method,threshold_strategy,F1,Precision,Recall,ROC_AUC,AUPRC,notes
1,FraudAutoEncoder,32,0.2,MSELoss,100,0.000723,mean_MSE,max_F1_on_valid,0.3077,0.3006,0.3151,0.7735,0.5471,0.2360,baseline v1 MSE | test_F1=0.2284 test_AUC=0.7625 test_Gini=0.5249
2,FraudAutoEncoderV2,64,0.05,MSELoss,150,0.000260,mean_MSE,max_F1_on_valid,0.3007,0.2514,0.3741,0.7729,0.5458,0.2060,bottleneck=64 dropout=0.05 MSE | test_F1=0.2403 test_AUC=0.7618 test_Gini=0.5235
4,FraudAutoEncoder,32,0.2,MSELoss,100,0.000723,mean_MAE,max_F1_on_valid,0.3240,0.3172,0.3312,0.7834,0.5669,0.2542,baseline v1 MAE | test_F1=0.2514 test_AUC=0.7725 test_Gini=0.5449
5,FraudAutoEncoderV2,64,0.05,MSELoss,150,0.000260,mean_MAE,max_F1_on_valid,0.3381,0.3218,0.3561,0.7823,0.5645,0.2637,bottleneck=64 dropout=0.05 MAE | test_F1=0.2619 test_AUC=0.7714 test_Gini=0.5429
3,FraudMLP,-,0.15,BCEWithLogitsLoss+pos_weight,20,0.493872,sigmoid_prob,max_F1_on_valid,0.5139,0.5960,0.4517,0.8914,0.7828,0.5271,supervised MLP benchmark | 451→256→128→64→1 LN+dropout=0.15 pos_w_cap10 ES=AUC | test_F1=0.4486 test_AUC=0.8662 test_Gini=0.7324
3,FraudMLP,-,0.15,BCEWithLogitsLoss+pos_weight,20,0.493872,sigmoid_prob,max_F1_on_valid,0.5139,0.5960,0.4517,0.8914,0.7828,0.5271,supervised MLP benchmark | 451→256→128→64→1 LN+dropout=0.15 pos_w_cap10 ES=AUC | test_F1=0.4486 test_AUC=0.8662 test_Gini=0.7324
6,FraudMLP,-,0.15,"FocalLoss(alpha=0.75,gamma=2.0)",37,0.015209,sigmoid_prob,max_F1_on_valid,0.5260,0.6141,0.4600,0.8927,0.7854,0.5272,MLP v2 | 451→512→256→128→1 FocalLoss lr=5e-4 patience=15 | test_F1=0.4485 test_AUC=0.8505 test_Gini=0.7009
7,FraudMLP,-,0.15,BCEWithLogitsLoss+pos_weight,26,0.468338,sigmoid_prob,max_F1_on_valid,0.5221,0.5847,0.4717,0.8932,0.7865,0.5349,supervised MLP v3 | 451→256→128→64→1 LN+dropout=0.15 pos_w_cap10 lr=5e-4 patience=15 ES=AUC | test_F1=0.4447 test_AUC=0.8610 test_Gini=0.7221
8,DAE+XGBoost,64,-,MSELoss(DAE)+XGB,150,0.002050,xgb_proba,max_F1_on_valid,0.4588,0.4792,0.4400,0.8674,0.7348,0.4543,DAE noise_rate=0.3 | XGB n_est=59 | aug=451+64+1 | test_F1=0.4045 test_AUC=0.8529 test_Gini=0.7058
